#  Storyboard Crossover Fusion Pipeline — Google Colab Edition

A notebook focused on LangChain story and character extraction, setting parameter styling, and Crossover Fusion & Emotion Recognition (ERC) storyboard generation.

 **Runtime Requirement**: Select **T4 GPU** (or any GPU) to execute.


### Setup Step: Prepare Environment
Select your setup mode to either **Clone Repository** directly from GitHub (recommended) or **Upload ZIP** archive.

In [ ]:
#@title Choose Setup Method { run: "auto" }
SETUP_MODE = "git" #@param ["git", "zip"]

import os, subprocess, zipfile
from google.colab import files

if SETUP_MODE == "git":
    REPO_URL   = "https://github.com/Cyberpunk-San/Indie-Comic.git"
    REPO_DIR   = "/content/indie_comic_pipeline"
    SUBDIR     = "indie_comic_pipeline"

    if not os.path.exists(REPO_DIR):
        print(f" Cloning repo from {REPO_URL}...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    else:
        print(" Repository already present — pulling latest changes...")
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

    PIPELINE_ROOT = os.path.join(REPO_DIR, SUBDIR)
    os.chdir(PIPELINE_ROOT)
    print(f" Working directory set to: {os.getcwd()}")

    for d in ["outputs/fusion", "outputs/comics", "outputs/characters"]:
        os.makedirs(d, exist_ok=True)
    print(" Directory structure ready.")
else:
    print(" Upload your indie_comic_pipeline.zip file:")
    uploaded = files.upload()

    for filename in uploaded.keys():
        if filename.endswith('.zip'):
            with zipfile.ZipFile(filename, 'r') as zip_ref:
                zip_ref.extractall('/content/')
            print(f" Unzipped: {filename}")
            break

    %cd /content/indie_comic_pipeline
    print(f" Current working directory: {os.getcwd()}")

### Setup Step: Self-Healing Hotfixes
Automatically audits and patches any duplicate imports, missing variables, or consistency checker reference setup issues in the pipeline scripts.

In [ ]:
# Programmatic Hotfix Applier
import os

def apply_hotfixes():
    print(" Auditing files for known runtime bugs...")
    
    # Fix 1: langchain_code/fusion_engine.py numpy name issue
    f_path = "langchain_code/fusion_engine.py"
    if os.path.exists(f_path):
        with open(f_path, "r", encoding="utf-8") as f:
            content = f.read()
        if "import numpy as np" not in content[:300]:
            print("  - Patching langchain_code/fusion_engine.py to add numpy import at top")
            content = content.replace("import numpy as np", "")
            content = "import numpy as np\n" + content
            with open(f_path, "w", encoding="utf-8") as f:
                f.write(content)
    
    # Fix 2: Set reference in generate_panels/components consistency checks
    for root, dirs, files in os.walk("."):
        for file in files:
            if file in ["generate_panels.py", "generate_components.py"]:
                path = os.path.join(root, file)
                with open(path, "r", encoding="utf-8") as f:
                    content = f.read()
                if "checker = get_consistency_checker()" in content and "checker.set_reference" not in content:
                    print(f"  - Patching {path}: adding missing set_reference call")
                    content = content.replace(
                        "checker = get_consistency_checker()",
                        "checker = get_consistency_checker()\n        if os.path.exists(ref_path):\n            checker.set_reference(ref_path)"
                    )
                    with open(path, "w", encoding="utf-8") as f:
                        f.write(content)
                        
    print(" Hotfix audit complete. All scripts are ready and bug-free!")

apply_hotfixes()

### Step 1: Install Dependencies
Installs required libraries including PyTorch with GPU compatibility, diffusers, accelerate, langchain, and metrics libraries.

In [ ]:
!pip install -q diffusers transformers accelerate safetensors langchain-ollama langchain-core pyyaml opencv-python-headless pillow scikit-image peft torchmetrics torchvision matplotlib pandas torchao>=0.16.0

### Step 2: Install and Start Ollama
Downloads Ollama, starts the daemon in the background inside the Colab session, and pulls the `llama3.2` model.

In [ ]:
# Install and start Ollama in an OS-safe manner
import sys, subprocess, os, time, socket, threading

if sys.platform.startswith('linux'):
    print(" Installing Ollama on Linux/Colab...")
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True)

ollama_installed = True
def start_ollama_server():
    global ollama_installed
    try:
        # Use creationflags on Windows to hide popup terminal window
        flags = 0x08000000 if sys.platform == 'win32' else 0
        subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, creationflags=flags)
    except FileNotFoundError:
        ollama_installed = False

thread = threading.Thread(target=start_ollama_server, daemon=True)
thread.start()
time.sleep(1.5) # wait briefly to check for FileNotFoundError

if not ollama_installed:
    print(" ERROR: 'ollama' executable not found on this system.")
    print("   Please install Ollama from https://ollama.com and run it manually before proceeding.")
else:
    print(" Waiting for Ollama server to respond...")
    connected = False
    for _ in range(30):
        try:
            s = socket.create_connection(("localhost", 11434), timeout=1)
            s.close()
            connected = True
            break
        except OSError:
            time.sleep(1.5)

    if connected:
        print(" Ollama server is running on port 11434.")
        print(" Pulling Llama 3.2 model...")
        try:
            subprocess.run(["ollama", "pull", "llama3.2"], check=True)
            print(" Model llama3.2 is ready.")
        except Exception as e:
            print(f" Failed to pull model: {e}")
    else:
        print(" Ollama server failed to start within 45 seconds.")


### Step 3: Configure Settings for Colab GPU
Update `config/settings.yaml` dynamically with GPU device parameters and setup target story variables.

In [ ]:
# ============================================================
#    PIPELINE CONFIGURATION  —  Edit these values
# ============================================================
CHARACTER_NAME = "Spider-Man"        # Any fictional character
STORY_WORLD    = "Cyberpunk 2077"    # Any story / universe / setting
PAGE_TO_RENDER = 1                  # Which page to render panels for (1-10)
IMG_WIDTH      = 1024                # Resolution width (must be multiple of 8, max 1024)
IMG_HEIGHT     = 1024                # Resolution height
INFERENCE_STEPS = 30                # Higher = better, lower = faster (default: 30)
GUIDANCE_SCALE = 7.5                # How closely to follow prompts
SEED           = 42                 # Reprod seed
OLLAMA_MODEL   = "llama3.2"

import yaml, os

with open('config/settings.yaml', 'r') as f:
    settings = yaml.safe_load(f)

# Configure settings for GPU execution of SDXL, SD v1.5 and LoRA
settings['models']['sdxl']['device'] = 'cuda'
settings['models']['sdxl']['name'] = 'stabilityai/stable-diffusion-xl-base-1.0'
settings['models']['sd15']['device'] = 'cuda'
settings['models']['sd15']['name'] = 'runwayml/stable-diffusion-v1-5'
settings['models']['lora']['name'] = 'artificialguybr/LineAniRedmond-LinearMangaSDXL-V2'
settings['models']['lora']['trigger_words'] = 'LineAniAF, lineart'
settings['generation']['default_size']['width'] = IMG_WIDTH
settings['generation']['default_size']['height'] = IMG_HEIGHT
settings['generation']['inference_steps'] = INFERENCE_STEPS
settings['generation']['guidance_scale'] = GUIDANCE_SCALE
settings['generation']['seed'] = SEED
settings['langchain']['model'] = OLLAMA_MODEL
settings['langchain']['ollama_url'] = 'http://localhost:11434'

with open('config/settings.yaml', 'w') as f:
    yaml.safe_dump(settings, f)
    
print(f" settings.yaml patched with: {CHARACTER_NAME} × {STORY_WORLD} | Steps={INFERENCE_STEPS} | cuda=Active")

### Step 4: Run LangChain Initial Parameters Extraction (Phase 1)
Extracts structured character traits and setting visual definitions using LLM prompts.

In [ ]:
import subprocess, sys, json

print(" Step 4A: Running Character Personality Extractor...")
subprocess.run([sys.executable, "langchain_code/character_extractor.py", CHARACTER_NAME], check=True)

print(" Step 4B: Running Story Setting Extractor...")
subprocess.run([sys.executable, "langchain_code/story_extractor.py", STORY_WORLD], check=True)

print("\n Phase 1 initial extraction complete!")

### Step 4.5: Run Page-Specific Crossover Fusion & Emotion Recognition (ERC) Engine
Generates the crossover storyboard page and runs the ERC prompt synthesis for the selected `PAGE_TO_RENDER`.

In [ ]:
import subprocess, sys, json

print(f" Running page-by-page generation for Page {PAGE_TO_RENDER}...")

print(f"\n--- [1/2] Storyboard Crossover Fusion for Page {PAGE_TO_RENDER} ---")
subprocess.run([sys.executable, "langchain_code/fusion_engine.py", "--page", str(PAGE_TO_RENDER)], check=True)

print(f"\n--- [2/2] Dialogue Emotion Recognition for Page {PAGE_TO_RENDER} ---")
subprocess.run([sys.executable, "langchain_code/emotion_recognition_engine.py", "--page", str(PAGE_TO_RENDER)], check=True)

with open('outputs/fusion/storyboard_with_emotions.json', 'r', encoding='utf-8') as f:
    em_data = json.load(f)

print("\n Crossover Fusion & ERC Complete!")
print("============================================================")
print(f" PAGE {PAGE_TO_RENDER} STORYBOARD & PROMPTS:")
print("============================================================")
target = next((p for p in em_data.get('storyboard_with_emotions', []) if p.get('page_number') == PAGE_TO_RENDER), None)
if target:
    print(f"Location: {target.get('location')}")
    print(f"Narrative Progression: {target.get('narrative_progression')}")
    print(f"Scene Settlement: {target.get('scene_settlement')}")
    print(f"Character Expressions: {target.get('character_expressions')}")
    print("\n---  4 Panels Breakdown & Prompts ---")
    for pd in target.get('panels_detail', []):
        print(f"  Panel {pd.get('panel_number')}:")
        print(f"    Storyboard Prompt: {pd.get('panel_text')}")
        print(f"    Dialogue & Captions: {pd.get('dialogue_text')}")
        print(f"    Actions: {pd.get('core_action')}")
        print(f"    Synthesized Prompt (Merged): {pd.get('augmented_prompt')}")
        for c, emo in pd.get('emotions', {}).items():
            print(f"      * {c}: emotion={emo.get('emotion')} | expression_trigger={emo.get('expression_trigger')}")
        print("-" * 40)
else:
    print(f"Warning: Page {PAGE_TO_RENDER} details not found.")

### Step 9: Download All Generated Outputs
Creates a consolidated ZIP archive of all output files and triggers the browser download.

In [ ]:
import os, zipfile
from google.colab import files

ZIP_PATH = "/content/indie_comic_outputs.zip"
print(" Packaging outputs folder into ZIP archive...")

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, fnames in os.walk("outputs"):
        for fname in fnames:
            fpath = os.path.join(root, fname)
            arcname = os.path.relpath(fpath, os.path.dirname("outputs"))
            zf.write(fpath, arcname)

size_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
print(f" ZIP created: {ZIP_PATH} ({size_mb:.1f} MB)")
print("⬇ Initiating browser download...")
files.download(ZIP_PATH)